# Financial Operations Analytics
## Revenue Forecasting · Churn Prediction · Profitability Analysis

---

[![Python](https://img.shields.io/badge/Python-3.10+-blue?logo=python)](https://python.org)
[![Prophet](https://img.shields.io/badge/Prophet-1.1-orange)](https://facebook.github.io/prophet/)
[![XGBoost](https://img.shields.io/badge/XGBoost-2.0-green)](https://xgboost.ai)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

### Datasets Used
| Dataset | Source | Purpose |
|---------|--------|---------|
| IBM Telco Customer Churn | IBM / Kaggle (public) | Churn Prediction, LTV, Profitability |
| Verizon Revenue (yfinance) | Yahoo Finance API | Revenue Forecasting |

### Project Structure
```
1. Setup & Data Loading
2. Exploratory Data Analysis
3. Revenue Forecasting  (Prophet + SARIMA)
4. Churn Prediction     (LR · RF · XGBoost + SHAP)
5. Profitability Analysis (LTV · Cohorts · P&L)
6. Portfolio Export     (PNGs · HTML Report · GitHub README)
```

## 1 — Setup & Data Loading

In [ ]:
!pip install prophet yfinance xgboost shap plotly kaleido --quiet
print('All packages installed.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    mean_absolute_error, mean_squared_error
)
import xgboost as xgb
import shap

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from prophet import Prophet

np.random.seed(42)

OUTPUT_DIR = 'portfolio_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PALETTE = ['#2C3E50','#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6']
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def save_fig(name):
    path = f'{OUTPUT_DIR}/{name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  Saved → {path}')

print('Setup complete.')

In [ ]:
# IBM Telco Customer Churn Dataset
TELCO_URL = (
    'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d'
    '/master/data/Telco-Customer-Churn.csv'
)
telco = pd.read_csv(TELCO_URL)
telco.columns = telco.columns.str.strip().str.lower().str.replace(' ', '_')
telco['totalcharges'] = pd.to_numeric(telco['totalcharges'], errors='coerce')
telco['totalcharges'].fillna(telco['totalcharges'].median(), inplace=True)
telco['churn_flag'] = (telco['churn'] == 'Yes').astype(int)
telco['monthlycharges'] = telco['monthlycharges'].astype(float)
print(f'IBM Telco loaded: {telco.shape[0]:,} customers | Churn rate: {telco["churn_flag"].mean()*100:.1f}%')

In [ ]:
# Verizon Revenue via yfinance
ticker = yf.Ticker('VZ')
income = ticker.quarterly_income_stmt
rev_row = income.loc['Total Revenue'] if 'Total Revenue' in income.index else income.iloc[0]
rev_ts = rev_row.sort_index().reset_index()
rev_ts.columns = ['ds', 'y']
rev_ts['ds'] = pd.to_datetime(rev_ts['ds'])
rev_ts['y'] = rev_ts['y'] / 1e9
rev_ts = rev_ts.sort_values('ds').dropna().reset_index(drop=True)
rev_ts = rev_ts[rev_ts['y'] > 0]
print(f'Verizon revenue loaded: {len(rev_ts)} quarters')
print(f'Range: {rev_ts["ds"].min().date()} → {rev_ts["ds"].max().date()}')
rev_ts.tail()

## 2 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('IBM Telco — Customer Overview', fontsize=18, fontweight='bold', y=1.01)

churn_counts = telco['churn'].value_counts()
axes[0,0].pie(churn_counts, labels=['Retained','Churned'], autopct='%1.1f%%',
              colors=['#3498DB','#E74C3C'], startangle=90,
              wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0,0].set_title('Overall Churn Rate')

telco[telco['churn']=='No']['tenure'].hist(ax=axes[0,1], bins=30, color='#3498DB', alpha=0.7, label='Retained', density=True)
telco[telco['churn']=='Yes']['tenure'].hist(ax=axes[0,1], bins=30, color='#E74C3C', alpha=0.7, label='Churned', density=True)
axes[0,1].set_title('Tenure Distribution by Churn')
axes[0,1].set_xlabel('Tenure (months)')
axes[0,1].legend()

telco.boxplot(column='monthlycharges', by='churn', ax=axes[0,2])
plt.sca(axes[0,2])
plt.title('Monthly Charges by Churn')

contract_churn = telco.groupby('contract')['churn_flag'].mean() * 100
bars = axes[1,0].bar(contract_churn.index, contract_churn.values,
                     color=['#E74C3C','#F39C12','#2ECC71'], edgecolor='white')
for bar, val in zip(bars, contract_churn.values):
    axes[1,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                   f'{val:.1f}%', ha='center', fontweight='bold')
axes[1,0].set_title('Churn Rate by Contract Type')
axes[1,0].set_ylabel('Churn Rate (%)')

internet_churn = telco.groupby('internetservice')['churn_flag'].mean() * 100
axes[1,1].bar(internet_churn.index, internet_churn.values,
              color=['#9B59B6','#3498DB','#E74C3C'], edgecolor='white')
axes[1,1].set_title('Churn Rate by Internet Service')
axes[1,1].set_ylabel('Churn Rate (%)')

pay_churn = telco.groupby('paymentmethod')['churn_flag'].mean().sort_values() * 100
axes[1,2].barh(pay_churn.index, pay_churn.values, color='#2C3E50', edgecolor='white')
axes[1,2].set_title('Churn Rate by Payment Method')
axes[1,2].set_xlabel('Churn Rate (%)')

plt.tight_layout()
save_fig('01_customer_overview')
plt.show()

In [ ]:
num_cols = ['tenure','monthlycharges','totalcharges','churn_flag']
corr = telco[num_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Financial Metrics Analysis', fontsize=16, fontweight='bold')

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            mask=mask, ax=axes[0], linewidths=1)
axes[0].set_title('Correlation Matrix')

sns.scatterplot(data=telco.sample(1000, random_state=42),
                x='tenure', y='monthlycharges', hue='churn',
                palette={'Yes':'#E74C3C','No':'#3498DB'}, alpha=0.6, ax=axes[1])
axes[1].set_title('Tenure vs Monthly Charges (colored by Churn)')

plt.tight_layout()
save_fig('02_correlation_analysis')
plt.show()

In [ ]:
service_cols = ['phoneservice','multiplelines','onlinesecurity','onlinebackup',
                'deviceprotection','techsupport','streamingtv','streamingmovies']
churn_by_service = {}
for col in service_cols:
    temp = telco[telco[col].isin(['Yes','No'])].groupby(col)['churn_flag'].mean() * 100
    if 'Yes' in temp.index:
        churn_by_service[col.replace('_',' ').title()] = {'With': temp['Yes'], 'Without': temp['No']}

service_df = pd.DataFrame(churn_by_service).T.reset_index()
service_df.columns = ['Service','With','Without']
x = np.arange(len(service_df))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - w/2, service_df['With'],   w, label='With Service',    color='#3498DB', edgecolor='white')
ax.bar(x + w/2, service_df['Without'],w, label='Without Service', color='#E74C3C', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(service_df['Service'], rotation=30, ha='right')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate: With vs Without Each Service', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig('03_service_churn_analysis')
plt.show()

## 3 — Revenue Forecasting (Verizon Real Data)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Verizon Communications — Quarterly Revenue Analysis', fontsize=14, fontweight='bold')

axes[0].fill_between(rev_ts['ds'], rev_ts['y'], alpha=0.25, color='#2C3E50')
axes[0].plot(rev_ts['ds'], rev_ts['y'], color='#2C3E50', lw=2.5, marker='o', ms=5)
z = np.polyfit(range(len(rev_ts)), rev_ts['y'], 1)
axes[0].plot(rev_ts['ds'], np.poly1d(z)(range(len(rev_ts))), 'r--', lw=1.5, label='Trend')
axes[0].set_title('Quarterly Revenue (Billions USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}B'))
axes[0].legend()

rev_ts['qoq_growth'] = rev_ts['y'].pct_change() * 100
colors_bar = ['#E74C3C' if v < 0 else '#2ECC71' for v in rev_ts['qoq_growth'].fillna(0)]
axes[1].bar(rev_ts['ds'], rev_ts['qoq_growth'].fillna(0), color=colors_bar, width=80, edgecolor='white')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Quarter-over-Quarter Revenue Growth (%)')

plt.tight_layout()
save_fig('04_revenue_trend')
plt.show()

In [ ]:
n_test = 4
n_forecast = 4
train_rev = rev_ts.iloc[:-n_test]
test_rev  = rev_ts.iloc[-n_test:]

# Prophet
prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                  daily_seasonality=False, seasonality_mode='multiplicative',
                  changepoint_prior_scale=0.05, interval_width=0.90)
prophet.fit(train_rev[['ds','y']])
future_df    = prophet.make_future_dataframe(periods=n_test + n_forecast, freq='QS')
prophet_pred = prophet.predict(future_df)
prophet_test = prophet_pred[prophet_pred['ds'].isin(test_rev['ds'])]['yhat'].values
p_mape = np.mean(np.abs((test_rev['y'].values - prophet_test) / test_rev['y'].values)) * 100

# SARIMA
sarima = SARIMAX(train_rev['y'], order=(1,1,1), seasonal_order=(1,0,0,4),
                 enforce_stationarity=False, enforce_invertibility=False)
sarima_fit  = sarima.fit(disp=False)
sarima_pred = sarima_fit.get_forecast(steps=n_test + n_forecast)
sarima_mean = sarima_pred.predicted_mean
sarima_ci   = sarima_pred.conf_int(alpha=0.10)
s_mape = np.mean(np.abs((test_rev['y'].values - sarima_mean.values[:n_test]) / test_rev['y'].values)) * 100

print(f'Prophet MAPE: {p_mape:.2f}% | SARIMA MAPE: {s_mape:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Verizon Revenue Forecast — SARIMA vs Prophet', fontsize=15, fontweight='bold')

sarima_dates = pd.date_range(test_rev['ds'].iloc[0], periods=n_test + n_forecast, freq='QS')
fc_p = prophet_pred[prophet_pred['ds'] > train_rev['ds'].iloc[-1]]

for ax, title, sarima_flag in [(axes[0], f'SARIMA (MAPE={s_mape:.1f}%)', True),
                                (axes[1], f'Prophet (MAPE={p_mape:.1f}%)', False)]:
    ax.plot(rev_ts['ds'], rev_ts['y'], color='#2C3E50', lw=2.5, label='Actual', marker='o', ms=4)
    ax.axvline(train_rev['ds'].iloc[-1], ls=':', color='gray', lw=1.5, label='Train/Test split')
    ax.scatter(test_rev['ds'], test_rev['y'], color='orange', zorder=5, s=60, label='Test Actual')
    if sarima_flag:
        ax.plot(sarima_dates, sarima_mean.values, color='#E74C3C', lw=2, ls='--', label='Forecast')
        ax.fill_between(sarima_dates, sarima_ci.iloc[:,0], sarima_ci.iloc[:,1], alpha=0.15, color='#E74C3C')
    else:
        ax.plot(fc_p['ds'], fc_p['yhat'], color='#2ECC71', lw=2, ls='--', label='Forecast')
        ax.fill_between(fc_p['ds'], fc_p['yhat_lower'], fc_p['yhat_upper'], alpha=0.15, color='#2ECC71')
    ax.set_title(title, fontsize=13)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.0f}B'))
    ax.legend(fontsize=9)

plt.tight_layout()
save_fig('05_revenue_forecast')
plt.show()

In [ ]:
comp_fig = prophet.plot_components(prophet_pred)
comp_fig.suptitle('Prophet Decomposition — Trend & Seasonality', fontsize=13, fontweight='bold')
comp_fig.tight_layout()
comp_fig.savefig(f'{OUTPUT_DIR}/06_prophet_components.png', dpi=150, bbox_inches='tight')
print(f'  Saved → {OUTPUT_DIR}/06_prophet_components.png')
plt.show()

## 4 — Churn Prediction (IBM Telco)

In [ ]:
churn_model_df = telco.copy()
for col in ['partner','dependents','phoneservice','paperlessbilling']:
    churn_model_df[col+'_bin'] = (churn_model_df[col] == 'Yes').astype(int)

le = LabelEncoder()
cat_cols = ['gender','multiplelines','internetservice','onlinesecurity','onlinebackup',
            'deviceprotection','techsupport','streamingtv','streamingmovies',
            'contract','paymentmethod']
for col in cat_cols:
    churn_model_df[col+'_enc'] = le.fit_transform(churn_model_df[col].astype(str))

FEATURES = ['tenure','monthlycharges','totalcharges',
            'partner_bin','dependents_bin','phoneservice_bin','paperlessbilling_bin',
            'gender_enc','multiplelines_enc','internetservice_enc',
            'onlinesecurity_enc','onlinebackup_enc','deviceprotection_enc',
            'techsupport_enc','streamingtv_enc','streamingmovies_enc',
            'contract_enc','paymentmethod_enc']

X = churn_model_df[FEATURES]
y = churn_model_df['churn_flag']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, C=1.0, random_state=42), True),
    'Random Forest'      : (RandomForestClassifier(n_estimators=300, max_depth=10,
                                                    class_weight='balanced', random_state=42, n_jobs=-1), False),
    'XGBoost'            : (xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                               subsample=0.8,
                                               scale_pos_weight=(y==0).sum()/(y==1).sum(),
                                               eval_metric='logloss', random_state=42, verbosity=0), False)
}
results = {}
for name, (model, scaled) in models.items():
    Xtr = X_train_sc if scaled else X_train
    Xte = X_test_sc  if scaled else X_test
    model.fit(Xtr, y_train)
    proba = model.predict_proba(Xte)[:, 1]
    pred  = model.predict(Xte)
    auc   = roc_auc_score(y_test, proba)
    ap    = average_precision_score(y_test, proba)
    results[name] = {'model':model,'proba':proba,'pred':pred,'auc':auc,'ap':ap}
    print(f'{name:25s} | AUC: {auc:.4f} | Avg Precision: {ap:.4f}')

best_name = max(results, key=lambda k: results[k]['auc'])
print(f'\nBest model: {best_name}')

In [ ]:
COLORS = {'Logistic Regression':'#9B59B6','Random Forest':'#2ECC71','XGBoost':'#E74C3C'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Churn Prediction Model Comparison — IBM Telco Dataset', fontsize=14, fontweight='bold')

axes[0].plot([0,1],[0,1],'k--', lw=1)
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    axes[0].plot(fpr, tpr, lw=2.5, color=COLORS[name], label=f"{name} (AUC={res['auc']:.3f})")
axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curve')
axes[0].legend(fontsize=9)

baseline = y_test.mean()
axes[1].axhline(baseline, ls='--', color='gray', lw=1, label=f'Baseline ({baseline:.3f})')
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res['proba'])
    axes[1].plot(rec, prec, lw=2.5, color=COLORS[name], label=f"{name} (AP={res['ap']:.3f})")
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
axes[1].legend(fontsize=9)

plt.tight_layout()
save_fig('07_roc_pr_curves')
plt.show()

In [ ]:
best = results[best_name]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Best Model: {best_name}', fontsize=14, fontweight='bold')

cm = confusion_matrix(y_test, best['pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Retained','Churned'], yticklabels=['Retained','Churned'],
            linewidths=1, cbar=False)
axes[0].set(title='Confusion Matrix', ylabel='Actual', xlabel='Predicted')

m = best['model']
if hasattr(m, 'feature_importances_'):
    imp = pd.Series(m.feature_importances_, index=FEATURES).nlargest(15).sort_values()
    imp.plot(kind='barh', ax=axes[1], color='#2C3E50', edgecolor='white')
    axes[1].set_title('Top 15 Feature Importances')

plt.tight_layout()
save_fig('08_model_performance')
plt.show()
print(classification_report(y_test, best['pred'], target_names=['Retained','Churned']))

In [ ]:
xgb_model = results['XGBoost']['model']
explainer  = shap.Explainer(xgb_model, X_train)
shap_vals  = explainer(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('SHAP Explainability — XGBoost Churn Model', fontsize=14, fontweight='bold')
plt.sca(axes[0])
shap.plots.bar(shap_vals, max_display=12, show=False)
plt.sca(axes[1])
shap.plots.beeswarm(shap_vals, max_display=12, show=False)
plt.tight_layout()
save_fig('09_shap_explainability')
plt.show()

In [ ]:
all_proba = xgb_model.predict_proba(X)[:, 1]
churn_scored = telco.copy()
churn_scored['churn_probability'] = all_proba
churn_scored['risk_tier'] = pd.cut(churn_scored['churn_probability'],
    bins=[0,0.20,0.40,0.60,1.0], labels=['Low','Medium','High','Critical'])

risk_summary = churn_scored.groupby('risk_tier', observed=True).agg(
    customers=('customerid','count'),
    avg_monthly_rev=('monthlycharges','mean'),
    total_monthly_rev=('monthlycharges','sum'),
    actual_churn_rate=('churn_flag','mean')
).assign(revenue_share=lambda x: x['total_monthly_rev']/x['total_monthly_rev'].sum()*100)

print('=== Churn Risk Segmentation ===')
print(risk_summary.round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Churn Risk Segmentation', fontsize=14, fontweight='bold')
risk_colors = ['#2ECC71','#F39C12','#E67E22','#E74C3C']

axes[0].bar(risk_summary.index, risk_summary['customers'], color=risk_colors, edgecolor='white')
axes[0].set_title('Customer Count')
axes[1].bar(risk_summary.index, risk_summary['total_monthly_rev'], color=risk_colors, edgecolor='white')
axes[1].set_title('Monthly Revenue at Risk')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[2].bar(risk_summary.index, risk_summary['actual_churn_rate']*100, color=risk_colors, edgecolor='white')
axes[2].set_title('Actual Churn Rate in Segment (%)')

plt.tight_layout()
save_fig('10_churn_risk_segments')
plt.show()

## 5 — Profitability Analysis

In [ ]:
GROSS_MARGIN = 0.65
contract_churn_rates = {'Month-to-month': 0.0426, 'One year': 0.0113, 'Two year': 0.0028}

churn_scored['gross_profit_monthly'] = churn_scored['monthlycharges'] * GROSS_MARGIN
churn_scored['monthly_churn_rate']   = churn_scored['contract'].map(contract_churn_rates)
churn_scored['ltv'] = (churn_scored['gross_profit_monthly'] / churn_scored['monthly_churn_rate']).round(2)

ltv_contract = churn_scored.groupby('contract').agg(
    customers=('customerid','count'), avg_ltv=('ltv','mean'), median_ltv=('ltv','median')
).round(2)
print('=== LTV by Contract ===')
print(ltv_contract.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Customer Lifetime Value Analysis', fontsize=14, fontweight='bold')

bars = axes[0].bar(ltv_contract.index, ltv_contract['avg_ltv'],
                   color=['#E74C3C','#F39C12','#2ECC71'], edgecolor='white')
for bar, val in zip(bars, ltv_contract['avg_ltv']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                 f'${val:,.0f}', ha='center', fontweight='bold')
axes[0].set_title('Average LTV by Contract')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

sns.violinplot(data=churn_scored[churn_scored['ltv'] < churn_scored['ltv'].quantile(0.95)],
               x='contract', y='ltv', palette=['#E74C3C','#F39C12','#2ECC71'],
               ax=axes[1], inner='quartile')
axes[1].set_title('LTV Distribution by Contract')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.tight_layout()
save_fig('11_ltv_analysis')
plt.show()

In [ ]:
churn_scored['tenure_cohort'] = pd.cut(churn_scored['tenure'],
    bins=[0,6,12,24,36,48,72,100],
    labels=['0-6m','7-12m','13-24m','25-36m','37-48m','49-72m','72m+'])

cohort_pnl = churn_scored.groupby('tenure_cohort', observed=True).agg(
    customers=('customerid','count'),
    avg_monthly_rev=('monthlycharges','mean'),
    avg_ltv=('ltv','mean'),
    churn_rate=('churn_flag','mean'),
    avg_gross_profit=('gross_profit_monthly','mean')
).round(2)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Profitability by Customer Tenure Cohort', fontsize=15, fontweight='bold')

axes[0,0].plot(range(len(cohort_pnl)), cohort_pnl['avg_monthly_rev'], marker='o', ms=8, color='#3498DB', lw=2.5)
axes[0,0].fill_between(range(len(cohort_pnl)), cohort_pnl['avg_monthly_rev'], alpha=0.2, color='#3498DB')
axes[0,0].set_xticks(range(len(cohort_pnl)))
axes[0,0].set_xticklabels(cohort_pnl.index, rotation=20)
axes[0,0].set_title('Avg Monthly Revenue by Tenure')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.0f}'))

bar_colors = ['#E74C3C' if v > 0.15 else '#F39C12' if v > 0.05 else '#2ECC71' for v in cohort_pnl['churn_rate']]
axes[0,1].bar(range(len(cohort_pnl)), cohort_pnl['churn_rate']*100, color=bar_colors, edgecolor='white')
axes[0,1].set_xticks(range(len(cohort_pnl)))
axes[0,1].set_xticklabels(cohort_pnl.index, rotation=20)
axes[0,1].set_title('Churn Rate by Tenure Cohort')

axes[1,0].bar(range(len(cohort_pnl)), cohort_pnl['avg_ltv'], color='#2C3E50', edgecolor='white')
axes[1,0].set_xticks(range(len(cohort_pnl)))
axes[1,0].set_xticklabels(cohort_pnl.index, rotation=20)
axes[1,0].set_title('Average LTV by Tenure Cohort')
axes[1,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

revenue_retention = (cohort_pnl['avg_monthly_rev'] / cohort_pnl['avg_monthly_rev'].iloc[0] * 100)
axes[1,1].plot(range(len(cohort_pnl)), revenue_retention, marker='s', ms=7, color='#9B59B6', lw=2.5)
axes[1,1].axhline(100, ls='--', color='gray', lw=1)
axes[1,1].set_xticks(range(len(cohort_pnl)))
axes[1,1].set_xticklabels(cohort_pnl.index, rotation=20)
axes[1,1].set_title('Revenue Index vs Cohort 0-6m')

plt.tight_layout()
save_fig('12_cohort_profitability')
plt.show()

In [ ]:
seg_pnl = churn_scored.groupby(['internetservice','contract']).agg(
    customers=('customerid','count'),
    total_revenue=('monthlycharges','sum'),
    total_gp=('gross_profit_monthly','sum'),
    churn_rate=('churn_flag','mean')
).reset_index()
seg_pnl['gross_margin_pct'] = (seg_pnl['total_gp'] / seg_pnl['total_revenue'] * 100).round(1)

pivot_gm = seg_pnl.pivot(index='internetservice', columns='contract', values='gross_margin_pct').fillna(0)
pivot_rev = seg_pnl.pivot(index='internetservice', columns='contract', values='total_revenue').fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Segment P&L: Internet Service × Contract Type', fontsize=14, fontweight='bold')

pivot_rev.plot(kind='bar', ax=axes[0], color=['#E74C3C','#F39C12','#2ECC71'], edgecolor='white')
axes[0].set_title('Monthly Revenue by Segment')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].tick_params(axis='x', rotation=15)

sns.heatmap(pivot_gm, annot=True, fmt='.1f', cmap='RdYlGn', ax=axes[1],
            linewidths=1, vmin=50, vmax=80, cbar_kws={'label':'Gross Margin %'})
axes[1].set_title('Gross Margin % Heatmap')

plt.tight_layout()
save_fig('13_segment_pnl')
plt.show()

In [ ]:
CAC_MAP = {'No': 80, 'DSL': 150, 'Fiber optic': 250}
churn_scored['cac'] = churn_scored['internetservice'].map(CAC_MAP)
churn_scored['ltv_cac_ratio'] = (churn_scored['ltv'] / churn_scored['cac']).round(2)

ltv_cac = churn_scored.groupby(['internetservice','contract']).agg(
    avg_ltv=('ltv','mean'), avg_cac=('cac','first'), ltv_cac_ratio=('ltv_cac_ratio','mean')
).round(1)
pivot_ratio = ltv_cac['ltv_cac_ratio'].unstack().fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LTV:CAC Ratio (>3 = Healthy)', fontsize=13, fontweight='bold')

sns.heatmap(pivot_ratio, annot=True, fmt='.1f', cmap='RdYlGn', ax=axes[0],
            linewidths=1, vmin=0, vmax=30, cbar_kws={'label':'LTV:CAC'})
axes[0].set_title('LTV:CAC Heatmap')

sample = churn_scored.sample(500, random_state=42)
sc = axes[1].scatter(sample['churn_probability'], sample['ltv'],
    c=sample['monthlycharges'], cmap='viridis', alpha=0.6, s=40)
plt.colorbar(sc, ax=axes[1], label='Monthly Charges ($)')
axes[1].set_title('LTV vs Churn Probability')
axes[1].set_xlabel('Churn Probability')
axes[1].set_ylabel('Estimated LTV ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.tight_layout()
save_fig('14_ltv_cac_ratio')
plt.show()

## 6 — Executive Dashboard & Export

In [ ]:
best_auc = results[best_name]['auc']
avg_ltv  = churn_scored['ltv'].mean()
critical = (churn_scored['risk_tier'] == 'Critical').sum()
rev_at_risk = churn_scored[churn_scored['risk_tier']=='Critical']['monthlycharges'].sum()

print('━'*60)
print('      FINANCIAL OPERATIONS ANALYTICS — SUMMARY')
print('━'*60)
print(f'  Dataset        : IBM Telco ({len(telco):,} customers)')
print(f'  Forecast Data  : Verizon Communications (yfinance)')
print()
print(f'  Prophet MAPE   : {p_mape:.2f}%')
print(f'  SARIMA  MAPE   : {s_mape:.2f}%')
print(f'  Best Churn Model: {best_name} (AUC={best_auc:.3f})')
print(f'  Overall Churn  : {telco["churn_flag"].mean()*100:.1f}%')
print(f'  Avg LTV        : ${avg_ltv:,.0f}')
print(f'  Critical Risk  : {critical:,} customers')
print(f'  Revenue at Risk: ${rev_at_risk:,.0f}/month')
print('━'*60)

In [ ]:
# Interactive Plotly dashboard
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Verizon Revenue Trend', 'Prophet Forecast',
        'Churn Rate by Contract', 'ROC Curves',
        'LTV by Contract', 'LTV vs Churn Probability'
    )
)

fig.add_trace(go.Scatter(x=rev_ts['ds'], y=rev_ts['y'], name='Revenue',
    fill='tozeroy', line=dict(color='#2C3E50')), row=1, col=1)

fc_p = prophet_pred[prophet_pred['ds'] > train_rev['ds'].iloc[-1]]
fig.add_trace(go.Scatter(x=rev_ts['ds'], y=rev_ts['y'], showlegend=False,
    line=dict(color='#2C3E50')), row=1, col=2)
fig.add_trace(go.Scatter(x=fc_p['ds'], y=fc_p['yhat'], name='Prophet',
    line=dict(color='#2ECC71', dash='dash')), row=1, col=2)

cc = telco.groupby('contract')['churn_flag'].mean().reset_index()
fig.add_trace(go.Bar(x=cc['contract'], y=(cc['churn_flag']*100).round(1),
    name='Churn %', marker_color=['#E74C3C','#F39C12','#2ECC71']), row=1, col=3)

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    fig.add_trace(go.Scatter(x=fpr, y=tpr,
        name=f"{name[:3]} AUC={res['auc']:.3f}", mode='lines'), row=2, col=1)

ltv_c = churn_scored.groupby('contract')['ltv'].mean().reset_index()
fig.add_trace(go.Bar(x=ltv_c['contract'], y=ltv_c['ltv'].round(0),
    name='Avg LTV', marker_color=['#E74C3C','#F39C12','#2ECC71']), row=2, col=2)

s2 = churn_scored.sample(300, random_state=42)
fig.add_trace(go.Scatter(x=s2['churn_probability'], y=s2['ltv'],
    mode='markers', name='LTV vs Risk',
    marker=dict(color=s2['monthlycharges'], colorscale='Viridis', size=5, opacity=0.7)), row=2, col=3)

fig.update_layout(height=700, width=1300,
    title_text='Financial Operations Analytics — Executive Dashboard',
    title_font_size=18, template='plotly_white')

html_path = f'{OUTPUT_DIR}/dashboard_interactive.html'
fig.write_html(html_path, full_html=True, include_plotlyjs='cdn')
print(f'Dashboard saved → {html_path}')

try:
    fig.write_image(f'{OUTPUT_DIR}/00_executive_dashboard.png', width=1300, height=700, scale=2)
    print(f'PNG saved → {OUTPUT_DIR}/00_executive_dashboard.png')
except Exception as e:
    print(f'PNG export skipped: {e}')

fig.show()

In [ ]:
import zipfile, os
zip_path = 'financial_analytics_portfolio.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            zf.write(os.path.join(root, file))

print(f'ZIP created: {zip_path}')
print(f'Files in portfolio_output:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024
    print(f'  {f:<45} {size:>6.1f} KB')

try:
    from google.colab import files
    files.download(zip_path)
    print('Download started!')
except:
    print(f'ZIP ready at: {zip_path}')